# 14 — Reservoir correctors before and after permanent-tide harmonisation

**Scientific question.** How much does harmonising ATL13's tide-free heights to the
zero/mean-tide crust convention change the six empirical station correctors?

**Input** `icesat2-atl13-kakhovka/outputs/tables/egg2015_to_evrf2019_by_station.csv`
(READ ONLY — the ICESat-2 project is not modified).
**Output** `Fig04_*`, `Fig15_*`.

In [1]:
# --- provenance: environment, git SHA and input hashes -----------------------
import hashlib, json, subprocess, sys, platform
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd, geopandas as gpd, pyproj, rasterio, matplotlib
from swot_dnipro import config as CFG

np.random.seed(CFG.SEED)

def sha256(p, cap=64 * 1024 * 1024):
    p = Path(p)
    if not p.exists() or p.stat().st_size > cap:
        return f"(skipped: {'missing' if not p.exists() else 'too large'})"
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for c in iter(lambda: fh.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()[:16]

def git_sha():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                       cwd=CFG.ROOT, text=True).strip()
    except Exception:
        return "(not a git checkout)"

print("python      ", platform.python_version())
print("numpy       ", np.__version__)
print("pandas      ", pd.__version__)
print("geopandas   ", gpd.__version__)
print("matplotlib  ", matplotlib.__version__)
print("pyproj/PROJ ", pyproj.__version__, "/", pyproj.proj_version_str)
print("rasterio/GDAL", rasterio.__version__, "/", rasterio.__gdal_version__)
print("git SHA     ", git_sha())
print("seed        ", CFG.SEED)
print()
for p in [CFG.EGG2015_TIF, CFG.UA2019Z_ASC, CFG.CORRECTOR_BY_STATION,
          CFG.KHERSON_GAUGE_PARQUET, CFG.KHERSON_ATL13]:
    print(f"  {sha256(p)}  {Path(p).name}")


python       3.12.3
numpy        2.5.2
pandas       3.0.5
geopandas    1.1.4
matplotlib   3.11.1
pyproj/PROJ  3.7.2 / 9.5.1
rasterio/GDAL 1.5.1 / 3.12.4
git SHA      7d9580b
seed         42

  (skipped: too large)  egg_2015.tif
  297440243e2ed8d6  ua_2019z.asc
  92d9fd71a1c8da39  egg2015_to_evrf2019_by_station.csv
  3380b29a0d68016b  80805_yearbook.parquet
  1deb7b5cfd362bd7  kherson_atl13_pass_levels.parquet


In [2]:
d = pd.read_csv(CFG.FIGDATA / "Fig04_permanent_tide_correctors.csv")
s = pd.read_csv(CFG.FIGDATA / "Fig04_regional_median_summary.csv")
display(d[["name_en", "lat", "free2mean_m", "c_original_m", "c_harmonised_m", "delta_c_m",
           "empirical_nmad_m", "n_matchups"]])
display(s)

,name_en,lat,free2mean_m,c_original_m,c_harmonised_m,delta_c_m,empirical_nmad_m,n_matchups
0,Nova Kakhovka,46.775432,-0.035748,-0.127699,-0.091951,0.035748,0.053047,12
1,Velyka Lepetykha,47.175331,-0.037007,-0.189308,-0.152301,0.037007,0.060078,5
2,Blahovishchenka,47.463877,-0.037915,-0.149735,-0.111820,0.037915,0.049492,11
3,Nikopol,47.554451,-0.038200,-0.207813,-0.169613,0.038200,0.034089,8
4,Plavni,47.567110,-0.038240,-0.217321,-0.179082,0.038240,0.060157,8
5,Rozumivka,47.771208,-0.038881,-0.146252,-0.107371,0.038881,0.037642,9


,quantity,original_m,harmonised_m,shift_m,station_to_station_nmad_original_m,station_to_station_nmad_harmonised_m,n_stations
0,regional median c,-0.169521,-0.13206,0.037461,0.045635,0.04614,6


**Recommendation, not applied.** The regional median moves
`c = −0.1695 m → −0.1321 m`. The existing chain `H = h_ATL13 − ζ_EGG2015` mixes a
tide-free ellipsoidal height with a zero-tide quasigeoid; adding
`tide_earth_free2mean` removes that inconsistency. Do **not** mix the old and
harmonised values in one table.